# 06 — Self-Consistency V2

Bu notebook 04'te dondurulmuş deterministic ana cevapları korur ve her
`question_id × model_id` için stochastic tekrarlar üretir.

## V2 değişikliği: answer-type-aware canonicalization

V1 yalnız yüzeysel string normalization kullanıyordu. Bu nedenle:

```text
11
On bir
On bir.
```

ve:

```text
42.195 km
42,195 km
42,195 kilometre
```

gibi aynı cevabın biçimsel varyantları farklı answer sınıfları sayılabiliyordu.

V2'de iki katman ayrı tutulur:

```text
RAW stochastic answer
        ↓
basic surface normalization       → audit metric
        ↓
answer-type-aware canonicalization → ana self-consistency metric
```

Ana feature'lar canonicalized cevaplar üzerinden hesaplanır:

- `self_consistency`
- `unique_answer_ratio`
- `sample_answer_entropy_normalized`
- `deterministic_majority_agreement`
- `deterministic_sample_agreement_rate`

Ayrıca audit amacıyla yüzey-form metrikleri de saklanır:

- `surface_self_consistency`
- `surface_unique_answer_ratio`
- `surface_sample_answer_entropy_normalized`

## Leakage kuralı

Canonicalization yalnız şu bilgileri kullanır:

- model output text
- `answer_type`

Şunlar kesinlikle kullanılmaz:

- `reference_answer`
- `accepted_aliases`
- `manual_is_correct`

Dolayısıyla ground-truth leakage yoktur.

## Generation contract DEĞİŞMEDİ

```text
K = 5
seeds = [101, 202, 303, 404, 505]
temperature = 0.7
top_p = 0.9
max_new_tokens = 20
do_sample = True
```

Bu nedenle V1'de üretilmiş raw stochastic sample cache'i V2 tarafından yeniden kullanılır.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install "transformers==4.57.6" "accelerate>=1.3,<2" "bitsandbytes>=0.45" "huggingface_hub>=0.34"


In [ ]:
import gc, hashlib, json, math, os, platform, re, time, unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch, transformers, accelerate, bitsandbytes as bnb

from google.colab import userdata
from huggingface_hub import HfApi
from transformers import set_seed

EXPECTED_TRANSFORMERS_VERSION = "4.57.6"

if transformers.__version__ != EXPECTED_TRANSFORMERS_VERSION:
    raise RuntimeError(
        f"Beklenen transformers={EXPECTED_TRANSFORMERS_VERSION}; "
        f"mevcut={transformers.__version__}. Runtime'ı yeniden başlatın."
    )

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime gereklidir.")

PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_Confidence_Thesis")

RESULTS_STAGE04_DIR = PROJECT_ROOT / "results" / "stage04"
RESULTS_STAGE05_DIR = PROJECT_ROOT / "results" / "stage05"
MODEL_SELECTION_DIR = PROJECT_ROOT / "results" / "model_selection"

STAGE06_ROOT = PROJECT_ROOT / "runs" / "self_consistency"
CACHE_ROOT = STAGE06_ROOT / "cache"
EXPORT_ROOT = STAGE06_ROOT / "exports"
RESULTS_STAGE06_DIR = PROJECT_ROOT / "results" / "stage06"

for p in [CACHE_ROOT, EXPORT_ROOT, RESULTS_STAGE06_DIR]:
    p.mkdir(parents=True, exist_ok=True)

VALIDATED_DATASET_VERSION = "validated_v002"

STAGE04_COMPLETION_PATH = RESULTS_STAGE04_DIR / f"{VALIDATED_DATASET_VERSION}_stage04_complete.json"
STAGE05_COMPLETION_PATH = RESULTS_STAGE05_DIR / f"{VALIDATED_DATASET_VERSION}_stage05_complete.json"
STAGE05_CRITICAL_AUDIT_PATH = RESULTS_STAGE05_DIR / f"{VALIDATED_DATASET_VERSION}_stage05_critical_problems.csv"
STAGE05_FORMAT_AUDIT_PATH = RESULTS_STAGE05_DIR / f"{VALIDATED_DATASET_VERSION}_stage05_format_violations.csv"
SELECTION_MANIFEST_PATH = MODEL_SELECTION_DIR / "selected_models_v1.json"

def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def sha256_object(obj):
    return sha256_text(canonical_json(obj))

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def read_jsonl(path):
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def append_jsonl_record(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
        f.flush()
        os.fsync(f.fileno())

def atomic_write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    os.replace(tmp, path)

print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)


In [ ]:
# Önceki stage'leri doğrula
if not STAGE04_COMPLETION_PATH.exists():
    raise FileNotFoundError(STAGE04_COMPLETION_PATH)

if not STAGE05_COMPLETION_PATH.exists():
    raise FileNotFoundError(
        "Stage-05 completion manifest yok. 05'in completion hücresini çalıştırın."
    )

with open(STAGE04_COMPLETION_PATH, "r", encoding="utf-8") as f:
    stage04_manifest = json.load(f)

with open(STAGE05_COMPLETION_PATH, "r", encoding="utf-8") as f:
    stage05_manifest = json.load(f)

if stage04_manifest["dataset_version"] != VALIDATED_DATASET_VERSION:
    raise RuntimeError("Stage-04 dataset version mismatch.")

if stage05_manifest["dataset_version"] != VALIDATED_DATASET_VERSION:
    raise RuntimeError("Stage-05 dataset version mismatch.")

if STAGE05_CRITICAL_AUDIT_PATH.exists():
    critical_df = pd.read_csv(STAGE05_CRITICAL_AUDIT_PATH)
    if len(critical_df) != 0:
        raise RuntimeError("Stage-05 kritik extraction problemi var.")
    print("Stage-05 critical audit: PASS")

if STAGE05_FORMAT_AUDIT_PATH.exists():
    format_df = pd.read_csv(STAGE05_FORMAT_AUDIT_PATH)
    print("Stage-05 format violations:", len(format_df), "(non-critical, preserved)")

with open(SELECTION_MANIFEST_PATH, "r", encoding="utf-8") as f:
    selection_manifest = json.load(f)

selection_manifest_sha256 = file_sha256(SELECTION_MANIFEST_PATH)
selected_models = selection_manifest["selected_models"]

stage04_exports_by_model = {
    item["model_id"]: item
    for item in stage04_manifest["model_exports"]
}

stage04_rows_by_model = {}

for spec in selected_models:
    artifact = stage04_exports_by_model[spec["model_id"]]
    path = Path(artifact["export_file"])

    if file_sha256(path) != artifact["export_sha256"]:
        raise RuntimeError(f"Stage-04 SHA mismatch: {spec['model_id']}")

    rows = read_jsonl(path)

    if len(rows) != stage04_manifest["n_questions"]:
        raise RuntimeError(f"Stage-04 row count mismatch: {spec['model_id']}")

    stage04_rows_by_model[spec["model_id"]] = rows

display(pd.DataFrame(selected_models)[
    ["model_id", "short_name", "revision", "backend", "compute_dtype"]
])


In [ ]:
# Self-consistency generation + canonicalization contract

# ------------------------------------------------------------
# Generation contract: V1 ile BİREBİR aynı tutulur.
# Cache reuse bunun sayesinde mümkündür.
# ------------------------------------------------------------
SAMPLE_GENERATION_VERSION = "stochastic_t07_p09_v1"

SELF_CONSISTENCY_SEEDS = [
    101,
    202,
    303,
    404,
    505,
]

SELF_CONSISTENCY_K = len(
    SELF_CONSISTENCY_SEEDS
)

TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 20

SYSTEM_PROMPT = (
    "Türkçe kısa ve doğrudan olgusal cevap ver. "
    "Sadece cevabı yaz; açıklama veya gerekçe verme. "
    "Cevabı bilmiyorsan yalnızca 'Bilmiyorum.' yaz."
)

PROMPT_VERSION = (
    selection_manifest[
        "prompt_version"
    ]
)

PROMPT_TEXT_SHA256 = (
    sha256_text(
        SYSTEM_PROMPT
    )
)

# ------------------------------------------------------------
# Aggregation/canonicalization V2
# ------------------------------------------------------------
SELF_CONSISTENCY_VERSION = (
    "self_consistency_k5_v2_answer_type_canonical"
)

CANONICALIZATION_VERSION = (
    "answer_type_canonicalization_tr_v2"
)

CANONICALIZATION_SPEC = {
    "basic_text": (
        "NFKC + Turkish-aware lowercase + edge punctuation "
        "+ collapsed whitespace"
    ),
    "NUMBER": (
        "whole Turkish number words -> digits; "
        "numeric punctuation canonicalization"
    ),
    "NUMBER_WITH_UNIT": (
        "leading number canonicalization + conservative unit synonym mapping; "
        "units are never removed"
    ),
    "DATE": (
        "Turkish month-name/full numeric date canonicalization when parseable"
    ),
    "BINARY": (
        "whole-answer yes/no semantic form normalization"
    ),
    "ENTITY_AND_SHORT_PHRASE": (
        "basic conservative text normalization only"
    ),
    "reference_answer_used": False,
    "accepted_aliases_used": False,
}

CANONICALIZATION_SPEC_SHA256 = (
    sha256_object(
        CANONICALIZATION_SPEC
    )
)

AGGREGATION_CONTRACT = {
    "self_consistency_version": (
        SELF_CONSISTENCY_VERSION
    ),
    "canonicalization_version": (
        CANONICALIZATION_VERSION
    ),
    "canonicalization_spec_sha256": (
        CANONICALIZATION_SPEC_SHA256
    ),
    "majority_tie_break": (
        "highest_count_then_first_fixed_seed_occurrence"
    ),
    "entropy_normalizer": (
        "log(K)"
    ),
}

AGGREGATION_CONFIG_HASH = (
    sha256_object(
        AGGREGATION_CONTRACT
    )
)

# ------------------------------------------------------------
# Basic normalization
#
# IMPORTANT:
# Bu fonksiyon V1 ile aynı davranışı korur.
# Raw sample cache contract'ını değiştirmiyoruz.
# ------------------------------------------------------------
def normalize_answer(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    text = (
        text
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
        .strip()
    )

    text = re.sub(
        r"^[\s\.,;:!?\"'`()\[\]{}]+|"
        r"[\s\.,;:!?\"'`()\[\]{}]+$",
        "",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


# ------------------------------------------------------------
# Unknown/refusal behavior
# ------------------------------------------------------------
PURE_UNKNOWN_FORMS = {
    "bilmiyorum",
    "emin değilim",
    "bilinmiyor",
    "bilemiyorum",
}

UNKNOWN_PATTERNS = [
    r"\bbilmiyorum\b",
    r"\bemin değilim\b",
    r"\bbilinmiyor\b",
    r"\bbilemiyorum\b",
]

REFUSAL_PATTERNS = [
    r"\bcevap veremem\b",
    r"\byanıt veremem\b",
    r"\byanit veremem\b",
    r"\byardımcı olamam\b",
    r"\byardimci olamam\b",
    r"\bbilgi veremem\b",
]


def pure_unknown(text):
    return int(
        normalize_answer(text)
        in PURE_UNKNOWN_FORMS
    )


def contains_unknown(text):
    norm = normalize_answer(
        text
    )

    return int(
        any(
            re.search(
                pattern,
                norm,
            )
            for pattern
            in UNKNOWN_PATTERNS
        )
    )


def mixed_unknown(text):
    return int(
        contains_unknown(text) == 1
        and
        pure_unknown(text) == 0
    )


def refusal(text):
    norm = normalize_answer(
        text
    )

    return int(
        any(
            re.search(
                pattern,
                norm,
            )
            for pattern
            in REFUSAL_PATTERNS
        )
    )


# ------------------------------------------------------------
# Turkish number-word parser
# ------------------------------------------------------------
TR_NUMBER_SMALL = {
    "sıfır": 0,
    "bir": 1,
    "iki": 2,
    "üç": 3,
    "dört": 4,
    "beş": 5,
    "altı": 6,
    "yedi": 7,
    "sekiz": 8,
    "dokuz": 9,
    "on": 10,
    "yirmi": 20,
    "otuz": 30,
    "kırk": 40,
    "elli": 50,
    "altmış": 60,
    "yetmiş": 70,
    "seksen": 80,
    "doksan": 90,
}

TR_NUMBER_SCALES = {
    "bin": 1_000,
    "milyon": 1_000_000,
    "milyar": 1_000_000_000,
    "trilyon": 1_000_000_000_000,
}

TR_NUMBER_WORDS = (
    set(
        TR_NUMBER_SMALL
    )
    |
    {
        "yüz",
        *TR_NUMBER_SCALES.keys(),
        "eksi",
        "artı",
    }
)


def parse_turkish_integer_words(
    text,
):
    """
    Yalnız tüm ifade Türkçe sayı kelimelerinden oluşuyorsa parse eder.
    Örn:
      'on bir' -> 11
      'iki yüz kırk üç' -> 243
      'bin dokuz yüz' -> 1900
    """
    norm = normalize_answer(
        text
    )

    if not norm:
        return None

    tokens = norm.split()

    sign = 1

    if tokens and tokens[0] in {
        "eksi",
        "artı",
    }:
        if tokens[0] == "eksi":
            sign = -1
        tokens = tokens[1:]

    if not tokens:
        return None

    if not all(
        token in TR_NUMBER_WORDS
        for token in tokens
    ):
        return None

    total = 0
    current = 0

    for token in tokens:
        if token in {
            "eksi",
            "artı",
        }:
            return None

        if token in TR_NUMBER_SMALL:
            current += (
                TR_NUMBER_SMALL[
                    token
                ]
            )
            continue

        if token == "yüz":
            current = (
                1
                if current == 0
                else current
            )
            current *= 100
            continue

        if token in TR_NUMBER_SCALES:
            scale = (
                TR_NUMBER_SCALES[
                    token
                ]
            )

            current = (
                1
                if current == 0
                else current
            )

            total += (
                current
                *
                scale
            )

            current = 0
            continue

        return None

    return int(
        sign
        *
        (
            total
            +
            current
        )
    )


def parse_leading_turkish_number_words(
    text,
):
    """
    NUMBER_WITH_UNIT için yalnız baştaki maximal Türkçe sayı kelimesi
    dizisini parse eder.

    'on bir kilometre'
      -> ('11', 'kilometre')
    """
    norm = normalize_answer(
        text
    )

    tokens = norm.split()

    if not tokens:
        return None

    prefix = []

    for token in tokens:
        if token in TR_NUMBER_WORDS:
            prefix.append(
                token
            )
        else:
            break

    if not prefix:
        return None

    parsed = (
        parse_turkish_integer_words(
            " ".join(
                prefix
            )
        )
    )

    if parsed is None:
        return None

    remainder = " ".join(
        tokens[
            len(prefix):
        ]
    ).strip()

    return (
        str(
            parsed
        ),
        remainder,
    )


# ------------------------------------------------------------
# Numeric punctuation canonicalization
# ------------------------------------------------------------
NUMERIC_SURFACE_RE = re.compile(
    r"^[+-]?\d+(?:[\.,]\d+)*$"
)


def canonicalize_numeric_surface(
    text,
):
    """
    Locale punctuation varyantlarını konservatif biçimde eşler.

    - 42,195   -> 42.195
    - 42.195   -> 42.195
    - 299.792,458 -> 299792.458
    - 299,792.458 -> 299792.458

    Tek separator'ün decimal mi thousands mı olduğuna dair ground-truth
    çıkarımı yapılmaz; yalnız punctuation varyantları canonicalize edilir.
    """
    norm = normalize_answer(
        text
    )

    compact = re.sub(
        r"(?<=\d)\s+(?=\d)",
        "",
        norm,
    )

    if not NUMERIC_SURFACE_RE.fullmatch(
        compact
    ):
        return None

    sign = ""

    if compact.startswith(
        (
            "+",
            "-",
        )
    ):
        sign = (
            "-"
            if compact[0] == "-"
            else ""
        )
        compact = compact[1:]

    has_dot = "." in compact
    has_comma = "," in compact

    if (
        has_dot
        and
        has_comma
    ):
        last_dot = compact.rfind(
            "."
        )
        last_comma = compact.rfind(
            ","
        )

        decimal_sep = (
            "."
            if last_dot > last_comma
            else ","
        )

        thousands_sep = (
            ","
            if decimal_sep == "."
            else "."
        )

        compact = compact.replace(
            thousands_sep,
            "",
        )

        compact = compact.replace(
            decimal_sep,
            ".",
        )

    elif compact.count(
        "."
    ) > 1:
        compact = compact.replace(
            ".",
            "",
        )

    elif compact.count(
        ","
    ) > 1:
        compact = compact.replace(
            ",",
            "",
        )

    else:
        compact = compact.replace(
            ",",
            ".",
        )

    if "." not in compact:
        compact = str(
            int(
                compact
            )
        )
    else:
        integer_part, fractional_part = (
            compact.split(
                ".",
                1,
            )
        )

        integer_part = str(
            int(
                integer_part
            )
        )

        compact = (
            integer_part
            +
            "."
            +
            fractional_part
        )

    return (
        sign
        +
        compact
    )


# ------------------------------------------------------------
# Unit canonicalization
# ------------------------------------------------------------
UNIT_PHRASE_REPLACEMENTS = [
    (
        r"\bkilometre\s*/\s*saniye\b",
        "km/s",
    ),
    (
        r"\bkilometre\s*/\s*saat\b",
        "km/h",
    ),
    (
        r"\bmetre\s*/\s*saniye\s*(?:kare|²|\^2)\b",
        "m/s²",
    ),
    (
        r"\bmetre\s*/\s*saniye\b",
        "m/s",
    ),
    (
        r"\bderece\s+celsius\b",
        "°c",
    ),
    (
        r"\bcelsius\s+derece\b",
        "°c",
    ),
    (
        r"\bsantigrat\s+derece\b",
        "°c",
    ),
]

UNIT_TOKEN_MAP = {
    "kilometre": "km",
    "kilometreler": "km",
    "kilometer": "km",
    "kilometers": "km",
    "km": "km",

    "metre": "m",
    "metreler": "m",
    "meter": "m",
    "meters": "m",
    "m": "m",

    "santimetre": "cm",
    "centimetre": "cm",
    "cm": "cm",

    "milimetre": "mm",
    "millimetre": "mm",
    "mm": "mm",

    "kilogram": "kg",
    "kilogramlar": "kg",
    "kg": "kg",

    "gram": "g",
    "gramlar": "g",
    "g": "g",

    "litre": "l",
    "liter": "l",
    "l": "l",

    "mililitre": "ml",
    "millilitre": "ml",
    "ml": "ml",

    "saniye": "s",
    "saniyeler": "s",
    "sn": "s",
    "sec": "s",

    "dakika": "min",
    "dakikalar": "min",
    "dk": "min",
    "min": "min",

    "saat": "h",
    "saatler": "h",
    "sa": "h",
    "h": "h",

    "kelvin": "k",
    "k": "k",

    "newton": "n",
    "n": "n",
}


def canonicalize_unit_text(
    text,
):
    norm = normalize_answer(
        text
    )

    if not norm:
        return ""

    norm = (
        norm
        .replace(
            "m/s^2",
            "m/s²",
        )
        .replace(
            "m/s2",
            "m/s²",
        )
        .replace(
            "km/sa",
            "km/h",
        )
    )

    for pattern, replacement in (
        UNIT_PHRASE_REPLACEMENTS
    ):
        norm = re.sub(
            pattern,
            replacement,
            norm,
        )

    tokens = norm.split()

    tokens = [
        UNIT_TOKEN_MAP.get(
            token,
            token,
        )
        for token in tokens
    ]

    return " ".join(
        tokens
    )


# ------------------------------------------------------------
# DATE canonicalization
# ------------------------------------------------------------
TR_MONTHS = {
    "ocak": 1,
    "şubat": 2,
    "mart": 3,
    "nisan": 4,
    "mayıs": 5,
    "haziran": 6,
    "temmuz": 7,
    "ağustos": 8,
    "eylül": 9,
    "ekim": 10,
    "kasım": 11,
    "aralık": 12,
}


def canonicalize_date(
    text,
):
    norm = normalize_answer(
        text
    )

    # DD Month YYYY
    match = re.fullmatch(
        r"(\d{1,2})\s+("
        +
        "|".join(
            TR_MONTHS.keys()
        )
        +
        r")\s+(\d{4})",
        norm,
    )

    if match:
        day = int(
            match.group(1)
        )

        month = (
            TR_MONTHS[
                match.group(2)
            ]
        )

        year = int(
            match.group(3)
        )

        if (
            1 <= day <= 31
            and
            1 <= month <= 12
        ):
            return (
                f"{year:04d}-"
                f"{month:02d}-"
                f"{day:02d}"
            )

    # DD.MM.YYYY / DD-MM-YYYY / DD/MM/YYYY
    match = re.fullmatch(
        r"(\d{1,2})[\./-](\d{1,2})[\./-](\d{4})",
        norm,
    )

    if match:
        day = int(
            match.group(1)
        )

        month = int(
            match.group(2)
        )

        year = int(
            match.group(3)
        )

        if (
            1 <= day <= 31
            and
            1 <= month <= 12
        ):
            return (
                f"{year:04d}-"
                f"{month:02d}-"
                f"{day:02d}"
            )

    # Month YYYY
    match = re.fullmatch(
        r"("
        +
        "|".join(
            TR_MONTHS.keys()
        )
        +
        r")\s+(\d{4})",
        norm,
    )

    if match:
        month = (
            TR_MONTHS[
                match.group(1)
            ]
        )

        year = int(
            match.group(2)
        )

        return (
            f"{year:04d}-"
            f"{month:02d}"
        )

    # Year only
    if re.fullmatch(
        r"\d{4}",
        norm,
    ):
        return norm

    return norm


# ------------------------------------------------------------
# Answer-type-aware canonicalization
# ------------------------------------------------------------
def canonicalize_number_answer(
    text,
):
    norm = normalize_answer(
        text
    )

    word_value = (
        parse_turkish_integer_words(
            norm
        )
    )

    if word_value is not None:
        return str(
            word_value
        )

    numeric = (
        canonicalize_numeric_surface(
            norm
        )
    )

    if numeric is not None:
        return numeric

    return norm


def canonicalize_number_with_unit(
    text,
):
    norm = normalize_answer(
        text
    )

    # Önce doğrudan numeric prefix.
    match = re.match(
        r"^([+-]?\d+(?:[\.,]\d+)*)"
        r"(?:\s+|$)"
        r"(.*)$",
        norm,
    )

    if match:
        number_part = (
            canonicalize_numeric_surface(
                match.group(1)
            )
            or
            match.group(1)
        )

        remainder = (
            canonicalize_unit_text(
                match.group(2)
            )
        )

        return (
            number_part
            if not remainder
            else (
                number_part
                +
                " "
                +
                remainder
            )
        )

    # Sonra Turkish number-word prefix.
    word_prefix = (
        parse_leading_turkish_number_words(
            norm
        )
    )

    if word_prefix is not None:
        number_part, remainder = (
            word_prefix
        )

        remainder = (
            canonicalize_unit_text(
                remainder
            )
        )

        return (
            number_part
            if not remainder
            else (
                number_part
                +
                " "
                +
                remainder
            )
        )

    return canonicalize_unit_text(
        norm
    )


def canonicalize_binary(
    text,
):
    norm = normalize_answer(
        text
    )

    yes_forms = {
        "evet",
        "doğru",
        "dogru",
        "yes",
    }

    no_forms = {
        "hayır",
        "hayir",
        "yanlış",
        "yanlis",
        "no",
    }

    if norm in yes_forms:
        return "evet"

    if norm in no_forms:
        return "hayır"

    return norm


def canonicalize_answer(
    text,
    answer_type,
):
    """
    Ground-truth bağımsız, answer-type-aware canonicalization.
    """
    answer_type = (
        str(
            answer_type
        )
        .strip()
        .upper()
    )

    if answer_type == "NUMBER":
        return canonicalize_number_answer(
            text
        )

    if answer_type == "NUMBER_WITH_UNIT":
        return canonicalize_number_with_unit(
            text
        )

    if answer_type == "DATE":
        return canonicalize_date(
            text
        )

    if answer_type == "BINARY":
        return canonicalize_binary(
            text
        )

    # ENTITY_* ve SHORT_PHRASE:
    # alias çözümlemesi YOK; yalnız basic normalization.
    return normalize_answer(
        text
    )


print(
    "Generation K:",
    SELF_CONSISTENCY_K,
)

print(
    "Seeds:",
    SELF_CONSISTENCY_SEEDS,
)

print(
    "Canonicalization:",
    CANONICALIZATION_VERSION,
)

print(
    "Canonicalization SHA:",
    CANONICALIZATION_SPEC_SHA256,
)

print(
    "Aggregation hash:",
    AGGREGATION_CONFIG_HASH,
)


In [ ]:
# Model loader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
    BitsAndBytesConfig,
)

DTYPE_MAP = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}

def get_dtype(name):
    return DTYPE_MAP[name]

def quant_config(spec):
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=get_dtype(spec["compute_dtype"]),
    )

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def get_tokenizer(frontend):
    return frontend.tokenizer if hasattr(frontend, "tokenizer") else frontend

def get_hf_token_if_needed(spec):
    if spec["backend"] != "gemma3_processor":
        return None

    token = userdata.get("HF_TOKEN")

    if not token:
        raise RuntimeError("Colab Secrets içinde HF_TOKEN bulunamadı.")

    HfApi(token=token).model_info(
        spec["model_id"],
        revision=spec["revision"],
    )

    return token

def load_model(spec):
    clear_gpu()

    token = get_hf_token_if_needed(spec)
    model_id = spec["model_id"]
    revision = spec["revision"]

    print("Model:", model_id)
    print("Revision:", revision)
    print("Compute dtype:", spec["compute_dtype"])

    if spec["backend"] == "gemma3_processor":
        frontend = AutoProcessor.from_pretrained(
            model_id,
            revision=revision,
            token=token,
            trust_remote_code=False,
        )

        model = Gemma3ForConditionalGeneration.from_pretrained(
            model_id,
            revision=revision,
            token=token,
            quantization_config=quant_config(spec),
            device_map="auto",
            torch_dtype=get_dtype(spec["model_dtype"]),
        )
    else:
        frontend = AutoTokenizer.from_pretrained(
            model_id,
            revision=revision,
            trust_remote_code=spec["trust_remote_code"],
            use_fast=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            revision=revision,
            quantization_config=quant_config(spec),
            device_map="auto",
            trust_remote_code=spec["trust_remote_code"],
            torch_dtype=get_dtype(spec["model_dtype"]),
        )

    model.eval()

    tokenizer = get_tokenizer(frontend)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    if model.generation_config.pad_token_id is None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    print("Loaded:", model.__class__.__name__)
    return frontend, model


In [ ]:
# Prompt/input + stochastic sample
def build_inputs(frontend, model, spec, question):
    if spec["backend"] == "gemma3_processor":
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}],
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": question}],
            },
        ]

        inputs = frontend.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
    else:
        tokenizer = get_tokenizer(frontend)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ]

        if spec["disable_thinking"]:
            try:
                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            except TypeError:
                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
        else:
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

        inputs = tokenizer(prompt, return_tensors="pt")

    return {k: v.to(model.device) for k, v in inputs.items()}

def decode_generated(frontend, ids):
    tokenizer = get_tokenizer(frontend)
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

def eos_id_set(model):
    value = model.generation_config.eos_token_id

    if value is None:
        return set()

    if isinstance(value, int):
        return {int(value)}

    return {int(x) for x in value}

@torch.inference_mode()
def generate_one_sample(frontend, model, spec, question, seed):
    set_seed(int(seed))

    inputs = build_inputs(
        frontend,
        model,
        spec,
        question,
    )

    prompt_len = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        return_dict_in_generate=True,
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    generated_ids = out.sequences[0][prompt_len:]
    answer = decode_generated(frontend, generated_ids)

    eos_ids = eos_id_set(model)

    last_id = (
        int(generated_ids[-1].item())
        if len(generated_ids) > 0
        else None
    )

    ended_with_eos = int(
        last_id in eos_ids
        if last_id is not None
        else False
    )

    truncated = int(
        len(generated_ids) >= MAX_NEW_TOKENS
        and ended_with_eos == 0
    )

    return {
        "sample_seed": int(seed),
        "sample_answer": answer,
        "sample_answer_normalized": normalize_answer(answer),
        "sample_generated_token_count": int(len(generated_ids)),
        "sample_generation_seconds": float(elapsed),
        "sample_pure_unknown": pure_unknown(answer),
        "sample_contains_unknown": contains_unknown(answer),
        "sample_mixed_unknown": mixed_unknown(answer),
        "sample_refusal": refusal(answer),
        "sample_empty": int(len(answer.strip()) == 0),
        "sample_ended_with_eos": ended_with_eos,
        "sample_truncated": truncated,
    }


In [ ]:
# Sample cache contract + V2 aggregation

def sample_contract(
    spec,
):
    """
    IMPORTANT:
    V1 generation contract ile aynı.
    Bu hash'i değiştirmeyerek mevcut raw sample cache'ini reuse ediyoruz.
    """
    return {
        "task": (
            "self_consistency_sample"
        ),
        "sample_generation_version": (
            SAMPLE_GENERATION_VERSION
        ),
        "model_id": (
            spec[
                "model_id"
            ]
        ),
        "model_revision": (
            spec[
                "revision"
            ]
        ),
        "backend": (
            spec[
                "backend"
            ]
        ),
        "compute_dtype": (
            spec[
                "compute_dtype"
            ]
        ),
        "model_dtype": (
            spec[
                "model_dtype"
            ]
        ),
        "prompt_version": (
            PROMPT_VERSION
        ),
        "prompt_text_sha256": (
            PROMPT_TEXT_SHA256
        ),
        "max_new_tokens": (
            MAX_NEW_TOKENS
        ),
        "do_sample": True,
        "temperature": (
            TEMPERATURE
        ),
        "top_p": (
            TOP_P
        ),
    }


def sample_config_hash(
    spec,
):
    return sha256_object(
        sample_contract(
            spec
        )
    )


def sample_cache_key(
    stage04_row,
    spec,
    seed,
):
    return sha256_object({
        "stage04_inference_cache_key": (
            stage04_row[
                "inference_cache_key"
            ]
        ),
        "sample_config_hash": (
            sample_config_hash(
                spec
            )
        ),
        "seed": int(
            seed
        ),
    })


def entropy_from_counts(
    counts,
    total,
):
    if total <= 1:
        return 0.0

    probs = np.array(
        list(
            counts.values()
        ),
        dtype=float,
    ) / float(
        total
    )

    entropy = -float(
        np.sum(
            probs
            *
            np.log(
                probs
            )
        )
    )

    return float(
        entropy
        /
        math.log(
            total
        )
    )


def majority_with_fixed_seed_tie_break(
    values,
):
    counts = Counter(
        values
    )

    max_count = max(
        counts.values()
    )

    candidates = {
        value
        for value, count
        in counts.items()
        if count == max_count
    }

    majority = next(
        value
        for value
        in values
        if value in candidates
    )

    return (
        majority,
        int(
            max_count
        ),
        counts,
    )


def aggregate_samples(
    deterministic_answer,
    answer_type,
    sample_rows,
):
    if (
        len(
            sample_rows
        )
        != SELF_CONSISTENCY_K
    ):
        raise RuntimeError(
            "Aggregate için K sample gerekli."
        )

    ordered = sorted(
        sample_rows,
        key=lambda row: row[
            "sample_seed"
        ],
    )

    raw_answers = [
        row[
            "sample_answer"
        ]
        for row in ordered
    ]

    # ---------------------------------------------------------
    # Surface layer — audit only
    # ---------------------------------------------------------
    surface_answers = [
        normalize_answer(
            answer
        )
        for answer in raw_answers
    ]

    (
        surface_majority,
        surface_majority_count,
        surface_counts,
    ) = (
        majority_with_fixed_seed_tie_break(
            surface_answers
        )
    )

    # ---------------------------------------------------------
    # Canonical layer — MAIN consistency features
    # ---------------------------------------------------------
    canonical_answers = [
        canonicalize_answer(
            answer,
            answer_type,
        )
        for answer in raw_answers
    ]

    (
        canonical_majority,
        canonical_majority_count,
        canonical_counts,
    ) = (
        majority_with_fixed_seed_tie_break(
            canonical_answers
        )
    )

    majority_example = next(
        row[
            "sample_answer"
        ]
        for row, canonical
        in zip(
            ordered,
            canonical_answers,
        )
        if canonical
        ==
        canonical_majority
    )

    deterministic_surface = (
        normalize_answer(
            deterministic_answer
        )
    )

    deterministic_canonical = (
        canonicalize_answer(
            deterministic_answer,
            answer_type,
        )
    )

    deterministic_sample_matches = sum(
        int(
            answer
            ==
            deterministic_canonical
        )
        for answer
        in canonical_answers
    )

    changed_sample_count = sum(
        int(surface != canonical)
        for surface, canonical in zip(
            surface_answers,
            canonical_answers,
        )
    )

    surface_unique_count = len(
        surface_counts
    )

    canonical_unique_count = len(
        canonical_counts
    )

    return {
        "answer_type": (
            answer_type
        ),

        "canonicalization_version": (
            CANONICALIZATION_VERSION
        ),

        "canonicalization_spec_sha256": (
            CANONICALIZATION_SPEC_SHA256
        ),

        # -----------------------------------------------------
        # MAIN canonical consistency features
        # -----------------------------------------------------
        "self_consistency": float(
            canonical_majority_count
            /
            SELF_CONSISTENCY_K
        ),

        "majority_answer": (
            majority_example
        ),

        "majority_answer_canonical": (
            canonical_majority
        ),

        "majority_count": int(
            canonical_majority_count
        ),

        "unique_answer_count": int(
            canonical_unique_count
        ),

        "unique_answer_ratio": float(
            canonical_unique_count
            /
            SELF_CONSISTENCY_K
        ),

        "sample_answer_entropy_normalized": (
            entropy_from_counts(
                canonical_counts,
                SELF_CONSISTENCY_K,
            )
        ),

        "deterministic_answer_canonical": (
            deterministic_canonical
        ),

        "deterministic_majority_agreement": int(
            deterministic_canonical
            ==
            canonical_majority
        ),

        "deterministic_sample_agreement_rate": float(
            deterministic_sample_matches
            /
            SELF_CONSISTENCY_K
        ),

        # -----------------------------------------------------
        # Surface-form audit metrics
        # -----------------------------------------------------
        "surface_self_consistency": float(
            surface_majority_count
            /
            SELF_CONSISTENCY_K
        ),

        "surface_majority_answer_normalized": (
            surface_majority
        ),

        "surface_unique_answer_count": int(
            surface_unique_count
        ),

        "surface_unique_answer_ratio": float(
            surface_unique_count
            /
            SELF_CONSISTENCY_K
        ),

        "surface_sample_answer_entropy_normalized": (
            entropy_from_counts(
                surface_counts,
                SELF_CONSISTENCY_K,
            )
        ),

        "canonicalization_changed_sample_count": int(
            changed_sample_count
        ),

        "canonicalization_reduced_unique_count": int(
            surface_unique_count
            -
            canonical_unique_count
        ),

        # -----------------------------------------------------
        # Behavioral sample rates
        # -----------------------------------------------------
        "unknown_sample_rate": float(
            np.mean(
                [
                    row[
                        "sample_pure_unknown"
                    ]
                    for row
                    in ordered
                ]
            )
        ),

        "contains_unknown_sample_rate": float(
            np.mean(
                [
                    row[
                        "sample_contains_unknown"
                    ]
                    for row
                    in ordered
                ]
            )
        ),

        "mixed_unknown_sample_rate": float(
            np.mean(
                [
                    row[
                        "sample_mixed_unknown"
                    ]
                    for row
                    in ordered
                ]
            )
        ),

        "refusal_sample_rate": float(
            np.mean(
                [
                    row[
                        "sample_refusal"
                    ]
                    for row
                    in ordered
                ]
            )
        ),

        "empty_sample_rate": float(
            np.mean(
                [
                    row[
                        "sample_empty"
                    ]
                    for row
                    in ordered
                ]
            )
        ),

        "sample_truncation_rate": float(
            np.mean(
                [
                    row[
                        "sample_truncated"
                    ]
                    for row
                    in ordered
                ]
            )
        ),

        "mean_sample_generation_seconds": float(
            np.mean(
                [
                    row[
                        "sample_generation_seconds"
                    ]
                    for row
                    in ordered
                ]
            )
        ),

        # -----------------------------------------------------
        # Audit payload
        # -----------------------------------------------------
        "sample_seeds": [
            row[
                "sample_seed"
            ]
            for row
            in ordered
        ],

        "sample_answers": (
            raw_answers
        ),

        "sample_answers_surface_normalized": (
            surface_answers
        ),

        "sample_answers_canonical": (
            canonical_answers
        ),
    }


In [ ]:
# V2 canonicalization unit tests — FULL RUN'DAN ÖNCE çalıştır

canonicalization_tests = [
    # NUMBER
    (
        "11",
        "NUMBER",
        "11",
    ),
    (
        "On bir",
        "NUMBER",
        "11",
    ),
    (
        "On bir.",
        "NUMBER",
        "11",
    ),
    (
        "Dört.",
        "NUMBER",
        "4",
    ),

    # NUMBER_WITH_UNIT
    (
        "42.195 km",
        "NUMBER_WITH_UNIT",
        "42.195 km",
    ),
    (
        "42,195 km",
        "NUMBER_WITH_UNIT",
        "42.195 km",
    ),
    (
        "42,195 kilometre",
        "NUMBER_WITH_UNIT",
        "42.195 km",
    ),
    (
        "1852 metre",
        "NUMBER_WITH_UNIT",
        "1852 m",
    ),
    (
        "on bir kilometre",
        "NUMBER_WITH_UNIT",
        "11 km",
    ),

    # DATE
    (
        "1 Kasım 1932",
        "DATE",
        "1932-11-01",
    ),
    (
        "01.11.1932",
        "DATE",
        "1932-11-01",
    ),

    # BINARY
    (
        "Evet.",
        "BINARY",
        "evet",
    ),
    (
        "Hayır.",
        "BINARY",
        "hayır",
    ),

    # ENTITY: conservative, no alias resolution
    (
        "NASA",
        "ENTITY_ORGANIZATION",
        "nasa",
    ),
]

test_rows = []

for raw, answer_type, expected in (
    canonicalization_tests
):
    actual = (
        canonicalize_answer(
            raw,
            answer_type,
        )
    )

    test_rows.append({
        "raw": raw,
        "answer_type": (
            answer_type
        ),
        "expected": (
            expected
        ),
        "actual": (
            actual
        ),
        "pass": int(
            actual
            ==
            expected
        ),
    })

test_df = pd.DataFrame(
    test_rows
)

display(
    test_df
)

if not (
    test_df[
        "pass"
    ]
    == 1
).all():
    raise RuntimeError(
        "Canonicalization unit test başarısız."
    )

print(
    "V2 CANONICALIZATION UNIT TESTS BAŞARILI"
)


# Kullanıcının fark ettiği TRQA000104 tipi örnek:
demo_samples = [
    {
        "sample_seed": 101,
        "sample_answer": "11",
        "sample_pure_unknown": 0,
        "sample_contains_unknown": 0,
        "sample_mixed_unknown": 0,
        "sample_refusal": 0,
        "sample_empty": 0,
        "sample_truncated": 0,
        "sample_generation_seconds": 0.0,
    },
    {
        "sample_seed": 202,
        "sample_answer": "On bir",
        "sample_pure_unknown": 0,
        "sample_contains_unknown": 0,
        "sample_mixed_unknown": 0,
        "sample_refusal": 0,
        "sample_empty": 0,
        "sample_truncated": 0,
        "sample_generation_seconds": 0.0,
    },
    {
        "sample_seed": 303,
        "sample_answer": "11",
        "sample_pure_unknown": 0,
        "sample_contains_unknown": 0,
        "sample_mixed_unknown": 0,
        "sample_refusal": 0,
        "sample_empty": 0,
        "sample_truncated": 0,
        "sample_generation_seconds": 0.0,
    },
    {
        "sample_seed": 404,
        "sample_answer": "On bir.",
        "sample_pure_unknown": 0,
        "sample_contains_unknown": 0,
        "sample_mixed_unknown": 0,
        "sample_refusal": 0,
        "sample_empty": 0,
        "sample_truncated": 0,
        "sample_generation_seconds": 0.0,
    },
    {
        "sample_seed": 505,
        "sample_answer": "On bir",
        "sample_pure_unknown": 0,
        "sample_contains_unknown": 0,
        "sample_mixed_unknown": 0,
        "sample_refusal": 0,
        "sample_empty": 0,
        "sample_truncated": 0,
        "sample_generation_seconds": 0.0,
    },
]

demo = aggregate_samples(
    deterministic_answer="11",
    answer_type="NUMBER",
    sample_rows=demo_samples,
)

print(
    "\n11 / On bir demo"
)

print(
    "Surface self-consistency:",
    demo[
        "surface_self_consistency"
    ],
)

print(
    "Canonical self-consistency:",
    demo[
        "self_consistency"
    ],
)

print(
    "Surface entropy:",
    demo[
        "surface_sample_answer_entropy_normalized"
    ],
)

print(
    "Canonical entropy:",
    demo[
        "sample_answer_entropy_normalized"
    ],
)

print(
    "Canonical samples:",
    demo[
        "sample_answers_canonical"
    ],
)


In [ ]:
# Incremental Stage-06 V2 model run

def run_stage06_for_model(
    spec,
):
    model_id = (
        spec[
            "model_id"
        ]
    )

    short_name = (
        spec[
            "short_name"
        ]
    )

    stage04_rows = (
        stage04_rows_by_model[
            model_id
        ]
    )

    # ---------------------------------------------------------
    # RAW SAMPLE contract — unchanged from V1
    # ---------------------------------------------------------
    contract = (
        sample_contract(
            spec
        )
    )

    config_hash = (
        sample_config_hash(
            spec
        )
    )

    config_short = (
        config_hash[
            :12
        ]
    )

    # ---------------------------------------------------------
    # AGGREGATION contract — NEW in V2
    # ---------------------------------------------------------
    aggregation_short = (
        AGGREGATION_CONFIG_HASH[
            :12
        ]
    )

    cache_dir = (
        CACHE_ROOT
        /
        short_name
    )

    cache_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    cache_path = (
        cache_dir
        /
        f"{config_short}.jsonl"
    )

    cache_lookup = {
        row[
            "sample_cache_key"
        ]: row
        for row
        in read_jsonl(
            cache_path
        )
    }

    pending = []

    for base in stage04_rows:
        for seed in (
            SELF_CONSISTENCY_SEEDS
        ):
            key = (
                sample_cache_key(
                    base,
                    spec,
                    seed,
                )
            )

            if (
                key
                not in cache_lookup
            ):
                pending.append({
                    "base": (
                        base
                    ),
                    "seed": int(
                        seed
                    ),
                    "key": (
                        key
                    ),
                })

    expected_sample_rows = (
        len(
            stage04_rows
        )
        *
        SELF_CONSISTENCY_K
    )

    reused = (
        expected_sample_rows
        -
        len(
            pending
        )
    )

    print(
        "\n========================================"
    )

    print(
        "MODEL:",
        model_id,
    )

    print(
        "Questions:",
        len(
            stage04_rows
        ),
    )

    print(
        "Expected raw samples:",
        expected_sample_rows,
    )

    print(
        "RAW SAMPLE cache reuse:",
        reused,
    )

    print(
        "New stochastic samples:",
        len(
            pending
        ),
    )

    print(
        "Sample config hash:",
        config_hash,
    )

    print(
        "Aggregation V2 hash:",
        AGGREGATION_CONFIG_HASH,
    )

    print(
        "========================================"
    )

    peak_gpu_gb = None

    # ---------------------------------------------------------
    # Yalnız cache'de eksik RAW sample varsa model yüklenir.
    # Gemma V1 600/600 cache mevcutsa model hiç yüklenmeyecek.
    # ---------------------------------------------------------
    if pending:
        frontend, model = (
            load_model(
                spec
            )
        )

        torch.cuda.reset_peak_memory_stats()

        for idx, item in enumerate(
            pending,
            start=1,
        ):
            base = (
                item[
                    "base"
                ]
            )

            result = (
                generate_one_sample(
                    frontend,
                    model,
                    spec,
                    base[
                        "question"
                    ],
                    item[
                        "seed"
                    ],
                )
            )

            record = {
                "sample_cache_key": (
                    item[
                        "key"
                    ]
                ),

                "sample_config_hash": (
                    config_hash
                ),

                "question_id": (
                    base[
                        "question_id"
                    ]
                ),

                "model_id": (
                    model_id
                ),

                "model_revision": (
                    spec[
                        "revision"
                    ]
                ),

                "stage04_inference_cache_key": (
                    base[
                        "inference_cache_key"
                    ]
                ),

                "question_text_sha256": (
                    base[
                        "question_text_sha256"
                    ]
                ),

                "created_at_utc": (
                    utc_now_iso()
                ),

                **result,
            }

            append_jsonl_record(
                cache_path,
                record,
            )

            cache_lookup[
                item[
                    "key"
                ]
            ] = (
                record
            )

            print(
                f"[{idx:04d}/{len(pending):04d}] "
                f"{base['question_id']} | "
                f"seed={item['seed']} | "
                f"{result['sample_answer']!r}"
            )

        peak_gpu_gb = float(
            torch.cuda.max_memory_allocated()
            /
            (1024 ** 3)
        )

        del model, frontend

        clear_gpu()

    else:
        print(
            "Tüm RAW stochastic samples cache'de. "
            "Model yüklenmedi."
        )

    # ---------------------------------------------------------
    # V2 aggregation — raw cache'den yeniden hesaplanır.
    # ---------------------------------------------------------
    export_rows = []

    for base in stage04_rows:
        samples = []

        for seed in (
            SELF_CONSISTENCY_SEEDS
        ):
            key = (
                sample_cache_key(
                    base,
                    spec,
                    seed,
                )
            )

            if (
                key
                not in cache_lookup
            ):
                raise RuntimeError(
                    f"Missing sample: "
                    f"{base['question_id']} seed={seed}"
                )

            samples.append(
                cache_lookup[
                    key
                ]
            )

        agg = (
            aggregate_samples(
                deterministic_answer=(
                    base[
                        "answer"
                    ]
                ),
                answer_type=(
                    base[
                        "answer_type"
                    ]
                ),
                sample_rows=(
                    samples
                ),
            )
        )

        export_rows.append({
            "dataset_version": (
                VALIDATED_DATASET_VERSION
            ),

            "question_id": (
                base[
                    "question_id"
                ]
            ),

            "model_id": (
                model_id
            ),

            "model_revision": (
                spec[
                    "revision"
                ]
            ),

            "stage04_inference_cache_key": (
                base[
                    "inference_cache_key"
                ]
            ),

            "frozen_deterministic_answer_sha256": (
                sha256_text(
                    base[
                        "answer"
                    ]
                )
            ),

            "self_consistency_version": (
                SELF_CONSISTENCY_VERSION
            ),

            "sample_config_hash": (
                config_hash
            ),

            "aggregation_config_hash": (
                AGGREGATION_CONFIG_HASH
            ),

            "K": (
                SELF_CONSISTENCY_K
            ),

            **agg,
        })

    export_dir = (
        EXPORT_ROOT
        /
        VALIDATED_DATASET_VERSION
    )

    export_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # V2 aggregation hash path'e dahil:
    # V1 immutable export ile çakışmaz.
    export_path = (
        export_dir
        /
        (
            f"{short_name}_"
            f"self_consistency_"
            f"{config_short}_"
            f"{aggregation_short}.jsonl"
        )
    )

    if not export_path.exists():
        atomic_write_jsonl(
            export_path,
            export_rows,
        )
    else:
        existing = (
            read_jsonl(
                export_path
            )
        )

        existing_keys = {
            (
                row[
                    "question_id"
                ],
                row[
                    "model_id"
                ],
            )
            for row
            in existing
        }

        expected_keys = {
            (
                row[
                    "question_id"
                ],
                row[
                    "model_id"
                ],
            )
            for row
            in export_rows
        }

        if existing_keys != expected_keys:
            raise RuntimeError(
                "Immutable Stage-06 V2 export key set mismatch."
            )

    export_sha = (
        file_sha256(
            export_path
        )
    )

    manifest = {
        "stage": (
            "06_self_consistency_v2"
        ),

        "dataset_version": (
            VALIDATED_DATASET_VERSION
        ),

        "model_id": (
            model_id
        ),

        "model_revision": (
            spec[
                "revision"
            ]
        ),

        "n_questions": (
            len(
                stage04_rows
            )
        ),

        "K": (
            SELF_CONSISTENCY_K
        ),

        "seeds": (
            SELF_CONSISTENCY_SEEDS
        ),

        "sample_contract": (
            contract
        ),

        "sample_config_hash": (
            config_hash
        ),

        "aggregation_contract": (
            AGGREGATION_CONTRACT
        ),

        "aggregation_config_hash": (
            AGGREGATION_CONFIG_HASH
        ),

        "canonicalization_spec": (
            CANONICALIZATION_SPEC
        ),

        "canonicalization_spec_sha256": (
            CANONICALIZATION_SPEC_SHA256
        ),

        "sample_cache_file": str(
            cache_path
        ),

        "sample_rows_expected": (
            expected_sample_rows
        ),

        "sample_rows_reused_this_run": (
            reused
        ),

        "sample_rows_new_this_run": (
            len(
                pending
            )
        ),

        "export_file": str(
            export_path
        ),

        "export_sha256": (
            export_sha
        ),

        "peak_gpu_memory_gb_for_new_inference": (
            peak_gpu_gb
        ),

        "created_at_utc": (
            utc_now_iso()
        ),
    }

    manifest_path = (
        export_dir
        /
        (
            f"{short_name}_"
            "stage06_v2_manifest.json"
        )
    )

    if not manifest_path.exists():
        with open(
            manifest_path,
            "x",
            encoding="utf-8",
        ) as f:
            json.dump(
                manifest,
                f,
                ensure_ascii=False,
                indent=2,
            )
    else:
        with open(
            manifest_path,
            "r",
            encoding="utf-8",
        ) as f:
            old_manifest = (
                json.load(
                    f
                )
            )

        if old_manifest["export_sha256"] != export_sha:
            raise RuntimeError(
                "Stage-06 V2 manifest/export SHA mismatch."
            )

    print(
        "\nSTAGE-06 V2 MODEL TAMAMLANDI"
    )

    print(
        "Export:",
        export_path,
    )

    return {
        "model_id": (
            model_id
        ),

        "raw_sample_cache_reused": (
            reused
        ),

        "new_raw_samples": (
            len(
                pending
            )
        ),

        "export": (
            export_path
        ),

        "manifest": (
            manifest_path
        ),
    }


## Model index ve run — V2

Her model için:

```text
120 soru × 5 sample = 600 raw stochastic generation
```

Gemma V1'de zaten çalıştırıldığı için V2'de Gemma için beklenen:

```text
RAW SAMPLE cache reuse: 600
New stochastic samples: 0
```

olmalıdır.

Bu durumda model hiç yüklenmeden yalnız V2 canonicalization + aggregation yapılır.

Qwen/Phi daha önce Stage-06'da çalıştırılmadıysa onların 600'er raw sample'ı normal şekilde
üretilecek.

Aynı model tekrar çalıştırılırsa sample-level cache nedeniyle generation tekrarlanmaz.


In [ ]:
MODEL_INDEX_TABLE = pd.DataFrame(
    [
        {
            "index": i,
            "model_id": s["model_id"],
            "short_name": s["short_name"],
            "compute_dtype": s["compute_dtype"],
        }
        for i, s in enumerate(selected_models)
    ]
)

display(MODEL_INDEX_TABLE)

MODEL_INDEX_TO_RUN = 0

stage06_run_result = run_stage06_for_model(
    selected_models[MODEL_INDEX_TO_RUN]
)

stage06_run_result


In [ ]:
# Üç model V2 export'unu birleştir ve audit et

combined_rows = []

missing = []

export_dir = (
    EXPORT_ROOT
    /
    VALIDATED_DATASET_VERSION
)

aggregation_short = (
    AGGREGATION_CONFIG_HASH[
        :12
    ]
)

for spec in selected_models:
    config_short = (
        sample_config_hash(
            spec
        )[
            :12
        ]
    )

    path = (
        export_dir
        /
        (
            f"{spec['short_name']}_"
            f"self_consistency_"
            f"{config_short}_"
            f"{aggregation_short}.jsonl"
        )
    )

    if not path.exists():
        missing.append(
            spec[
                "model_id"
            ]
        )
        continue

    combined_rows.extend(
        read_jsonl(
            path
        )
    )

if missing:
    raise RuntimeError(
        f"Eksik Stage-06 V2 modelleri: {missing}"
    )

combined06_df = (
    pd.DataFrame(
        combined_rows
    )
)

expected = (
    stage04_manifest[
        "n_questions"
    ]
    *
    len(
        selected_models
    )
)

print(
    "Stage-06 V2 rows:",
    len(
        combined06_df
    ),
)

print(
    "Expected:",
    expected,
)

if len(combined06_df) != expected:
    raise RuntimeError(
        "Stage-06 V2 coverage eksik."
    )

if combined06_df.duplicated(
    subset=[
        "question_id",
        "model_id",
    ]
).any():
    raise RuntimeError(
        "Duplicate question_id × model_id."
    )


# -------------------------------------------------------------
# Main audit
# -------------------------------------------------------------
audit = (
    combined06_df
    .groupby(
        "model_id"
    )
    .agg(
        n=(
            "question_id",
            "count",
        ),

        mean_self_consistency=(
            "self_consistency",
            "mean",
        ),

        median_self_consistency=(
            "self_consistency",
            "median",
        ),

        mean_unique_answer_ratio=(
            "unique_answer_ratio",
            "mean",
        ),

        mean_sample_entropy=(
            "sample_answer_entropy_normalized",
            "mean",
        ),

        deterministic_majority_agreement_rate=(
            "deterministic_majority_agreement",
            "mean",
        ),

        mean_deterministic_sample_agreement_rate=(
            "deterministic_sample_agreement_rate",
            "mean",
        ),

        mean_unknown_sample_rate=(
            "unknown_sample_rate",
            "mean",
        ),

        mean_contains_unknown_sample_rate=(
            "contains_unknown_sample_rate",
            "mean",
        ),

        mean_mixed_unknown_sample_rate=(
            "mixed_unknown_sample_rate",
            "mean",
        ),

        mean_refusal_sample_rate=(
            "refusal_sample_rate",
            "mean",
        ),

        mean_empty_sample_rate=(
            "empty_sample_rate",
            "mean",
        ),

        mean_sample_truncation_rate=(
            "sample_truncation_rate",
            "mean",
        ),

        # Surface-vs-canonical impact
        mean_surface_self_consistency=(
            "surface_self_consistency",
            "mean",
        ),

        mean_surface_entropy=(
            "surface_sample_answer_entropy_normalized",
            "mean",
        ),

        total_canonicalization_changed_samples=(
            "canonicalization_changed_sample_count",
            "sum",
        ),

        questions_where_unique_count_reduced=(
            "canonicalization_reduced_unique_count",
            lambda s: int(
                (
                    s > 0
                ).sum()
            ),
        ),
    )
    .reset_index()
)

display(
    audit
)


# -------------------------------------------------------------
# Canonicalization impact audit
# -------------------------------------------------------------
impact_df = combined06_df.loc[
    combined06_df[
        "canonicalization_reduced_unique_count"
    ] > 0,
    [
        "question_id",
        "model_id",
        "answer_type",
        "surface_self_consistency",
        "self_consistency",
        "surface_unique_answer_count",
        "unique_answer_count",
        "surface_sample_answer_entropy_normalized",
        "sample_answer_entropy_normalized",
        "sample_answers",
        "sample_answers_surface_normalized",
        "sample_answers_canonical",
    ],
].copy()

print(
    "Canonicalization'ın unique-answer count'u azalttığı question×model:",
    len(
        impact_df
    ),
)

display(
    impact_df.head(
        50
    )
)


AUDIT_PATH = (
    RESULTS_STAGE06_DIR
    /
    (
        f"{VALIDATED_DATASET_VERSION}"
        "_stage06_v2_audit.csv"
    )
)

CANONICALIZATION_IMPACT_PATH = (
    RESULTS_STAGE06_DIR
    /
    (
        f"{VALIDATED_DATASET_VERSION}"
        "_stage06_v2_canonicalization_impact.csv"
    )
)

audit.to_csv(
    AUDIT_PATH,
    index=False,
)

impact_df.to_csv(
    CANONICALIZATION_IMPACT_PATH,
    index=False,
)


# -------------------------------------------------------------
# Structural sanity
# -------------------------------------------------------------
bounded_columns = [
    "self_consistency",
    "unique_answer_ratio",
    "sample_answer_entropy_normalized",
    "deterministic_majority_agreement",
    "deterministic_sample_agreement_rate",

    "surface_self_consistency",
    "surface_unique_answer_ratio",
    "surface_sample_answer_entropy_normalized",

    "unknown_sample_rate",
    "contains_unknown_sample_rate",
    "mixed_unknown_sample_rate",
    "refusal_sample_rate",
    "empty_sample_rate",
    "sample_truncation_rate",
]

for col in bounded_columns:
    values = pd.to_numeric(
        combined06_df[
            col
        ],
        errors="coerce",
    )

    if values.isna().any():
        raise RuntimeError(
            f"Missing/non-numeric: {col}"
        )

    if (
        (values < -1e-12)
        |
        (values > 1 + 1e-12)
    ).any():
        raise RuntimeError(
            f"[0,1] dışında: {col}"
        )


# Canonicalization consistency asla surface consistency'den
# düşük olmamalı; canonicalization yalnız sınıfları birleştirebilir.
if (
    combined06_df[
        "self_consistency"
    ]
    +
    1e-12
    <
    combined06_df[
        "surface_self_consistency"
    ]
).any():
    raise RuntimeError(
        "Canonical self-consistency surface self-consistency'den düşük."
    )


if (
    combined06_df[
        "unique_answer_count"
    ]
    >
    combined06_df[
        "surface_unique_answer_count"
    ]
).any():
    raise RuntimeError(
        "Canonicalization unique-answer count'u artırmış."
    )


print(
    "STAGE-06 V2 STRUCTURAL SANITY CHECK BAŞARILI"
)


In [ ]:
# Düşük consistency / canonicalization / behavioral anomaliler

interesting_mask = (
    (
        combined06_df[
            "self_consistency"
        ]
        <= 0.4
    )
    |
    (
        combined06_df[
            "deterministic_majority_agreement"
        ]
        == 0
    )
    |
    (
        combined06_df[
            "contains_unknown_sample_rate"
        ]
        > 0
    )
    |
    (
        combined06_df[
            "sample_truncation_rate"
        ]
        > 0
    )
    |
    (
        combined06_df[
            "canonicalization_reduced_unique_count"
        ]
        > 0
    )
)

interesting_df = (
    combined06_df.loc[
        interesting_mask,
        [
            "question_id",
            "model_id",
            "answer_type",

            "surface_self_consistency",
            "self_consistency",

            "surface_unique_answer_count",
            "unique_answer_count",

            "surface_sample_answer_entropy_normalized",
            "sample_answer_entropy_normalized",

            "canonicalization_reduced_unique_count",

            "majority_answer",
            "majority_answer_canonical",

            "deterministic_answer_canonical",
            "deterministic_majority_agreement",
            "deterministic_sample_agreement_rate",

            "contains_unknown_sample_rate",
            "sample_truncation_rate",

            "sample_answers",
            "sample_answers_canonical",
        ],
    ]
    .sort_values(
        [
            "model_id",
            "self_consistency",
            "question_id",
        ]
    )
)

print(
    "Audit-worthy records:",
    len(
        interesting_df
    ),
)

display(
    interesting_df.head(
        100
    )
)

INTERESTING_PATH = (
    RESULTS_STAGE06_DIR
    /
    (
        f"{VALIDATED_DATASET_VERSION}"
        "_stage06_v2_interesting_records.csv"
    )
)

interesting_df.to_csv(
    INTERESTING_PATH,
    index=False,
)


In [ ]:
# Stage-06 V2 completion manifest

model_artifacts = []

export_dir = (
    EXPORT_ROOT
    /
    VALIDATED_DATASET_VERSION
)

for spec in selected_models:
    manifest_path = (
        export_dir
        /
        (
            f"{spec['short_name']}_"
            "stage06_v2_manifest.json"
        )
    )

    if not manifest_path.exists():
        raise RuntimeError(
            f"Stage-06 V2 manifest eksik: "
            f"{spec['model_id']}"
        )

    with open(
        manifest_path,
        "r",
        encoding="utf-8",
    ) as f:
        model_artifacts.append(
            json.load(
                f
            )
        )


STAGE06_COMPLETION_PATH = (
    RESULTS_STAGE06_DIR
    /
    (
        f"{VALIDATED_DATASET_VERSION}"
        "_stage06_v2_complete.json"
    )
)

completion = {
    "stage": (
        "06_self_consistency_v2"
    ),

    "dataset_version": (
        VALIDATED_DATASET_VERSION
    ),

    "stage04_completion_manifest_sha256": (
        file_sha256(
            STAGE04_COMPLETION_PATH
        )
    ),

    "stage05_completion_manifest_sha256": (
        file_sha256(
            STAGE05_COMPLETION_PATH
        )
    ),

    "selection_manifest_sha256": (
        selection_manifest_sha256
    ),

    "n_questions": (
        stage04_manifest[
            "n_questions"
        ]
    ),

    "n_models": (
        len(
            selected_models
        )
    ),

    "n_model_answer_rows": (
        len(
            combined06_df
        )
    ),

    "K": (
        SELF_CONSISTENCY_K
    ),

    "seeds": (
        SELF_CONSISTENCY_SEEDS
    ),

    "sample_generation_version": (
        SAMPLE_GENERATION_VERSION
    ),

    "self_consistency_version": (
        SELF_CONSISTENCY_VERSION
    ),

    "canonicalization_version": (
        CANONICALIZATION_VERSION
    ),

    "canonicalization_spec": (
        CANONICALIZATION_SPEC
    ),

    "canonicalization_spec_sha256": (
        CANONICALIZATION_SPEC_SHA256
    ),

    "aggregation_contract": (
        AGGREGATION_CONTRACT
    ),

    "aggregation_config_hash": (
        AGGREGATION_CONFIG_HASH
    ),

    "model_artifacts": (
        model_artifacts
    ),

    "audit_file": str(
        AUDIT_PATH
    ),

    "canonicalization_impact_file": str(
        CANONICALIZATION_IMPACT_PATH
    ),

    "interesting_records_file": str(
        INTERESTING_PATH
    ),

    "created_at_utc": (
        utc_now_iso()
    ),
}


if STAGE06_COMPLETION_PATH.exists():
    with open(
        STAGE06_COMPLETION_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        old = json.load(
            f
        )

    if old["aggregation_config_hash"] != AGGREGATION_CONFIG_HASH:
        raise RuntimeError(
            "Mevcut Stage-06 V2 completion farklı aggregation contract taşıyor."
        )

    print(
        "Stage-06 V2 completion zaten mevcut:"
    )

    print(
        STAGE06_COMPLETION_PATH
    )

else:
    with open(
        STAGE06_COMPLETION_PATH,
        "x",
        encoding="utf-8",
    ) as f:
        json.dump(
            completion,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(
        "STAGE 06 V2 DONDURULDU:"
    )

    print(
        STAGE06_COMPLETION_PATH
    )


# 06 V2 tamamlanma kriterleri

- [ ] Stage-04 frozen answers SHA ile doğrulandı
- [ ] Stage-05 completion doğrulandı
- [ ] Generation contract V1 ile aynı tutuldu
- [ ] Gemma V1 raw sample cache'i yeniden kullanıldı
- [ ] `answer_type` dışında ground-truth bilgisi canonicalization'a verilmedi
- [ ] NUMBER için Türkçe sayı-yazı varyantları canonicalize edildi
- [ ] NUMBER_WITH_UNIT için conservative number/unit canonicalization uygulandı
- [ ] DATE ve BINARY için konservatif canonicalization uygulandı
- [ ] ENTITY/SHORT_PHRASE için alias çözümlemesi yapılmadı
- [ ] Ana `self_consistency` canonical cevaplar üzerinden üretildi
- [ ] Surface-form consistency ayrıca audit metriği olarak saklandı
- [ ] `unique_answer_ratio` canonical cevaplar üzerinden üretildi
- [ ] `sample_answer_entropy_normalized` canonical cevaplar üzerinden üretildi
- [ ] `deterministic_majority_agreement` canonical cevaplar üzerinden üretildi
- [ ] unknown/refusal/truncation sample oranları korundu
- [ ] `sample_correct_rate` üretilmedi
- [ ] canonical consistency hiçbir satırda surface consistency'den düşük değil
- [ ] canonical unique count hiçbir satırda surface unique count'tan büyük değil
- [ ] 360 aggregate row tamamlandı
- [ ] `validated_vXXX_stage06_v2_complete.json` oluşturuldu

## Sonraki aşama — 07 Manual Annotation

07'de ilk kez target `manual_is_correct` oluşturulacak.

Confidence ve consistency sinyalleri annotator'a gösterilmeyecek; annotator yalnız:

- question
- frozen deterministic model answer
- reference answer
- accepted aliases

üzerinden karar verecek.
